# **KKBOX Churn Prediction And Retention Intelligence System - Modeling & Evaluation**

## **1. Objective**

## **2. Setup**

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report
)

from sklearn.ensemble import RandomForestClassifier

!pip install xgboost==1.6.2
from xgboost import XGBClassifier

!pip install shap
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

## **3. Load Clean Dataset**

In [ ]:
current_dir = os.getcwd()

while not os.path.exists(os.path.join(current_dir, "data", "processed")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        raise FileNotFoundError("Project root with data/processed not found")
    current_dir = parent


os.chdir(current_dir)

print("Project root:", os.getcwd())
print("data/processed exists:", os.path.exists("data/processed"))


pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

target = "is_churn"


df = pd.read_csv("data/processed/df_clean.csv")
df.head()

Project root: c:\Users\pauli\OneDrive\Documentos\GitHub\customer-churn-intelligence-system
data/processed exists: True


,msno,is_churn,gender,age,city_grouped,registered_via_grouped,avg_amount_paid,total_amount_paid,has_auto_renew,has_cancelled,total_secs,num_unq,customer_tenure_days,listening_group,payment_variability
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,male,28.0,5.0,3.0,0.0,0.0,0.0,0.0,80598.557,348.0,0.0,Medium-High,0.000000
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,male,20.0,13.0,3.0,180.0,180.0,0.0,0.0,6986.509,30.0,1174.0,Medium-Low,90.000000
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,male,18.0,13.0,3.0,150.0,300.0,0.0,0.0,67810.467,432.0,1173.0,Medium-High,100.000000
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,unknown,NaN,1.0,7.0,149.0,1490.0,1.0,0.0,0.000,0.0,698.0,Low,135.454545
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1,female,35.0,13.0,7.0,99.0,792.0,1.0,1.0,239882.241,548.0,1146.0,High,88.000000


## **4. Train/Test Split**

In [11]:
X = df.drop(columns=["is_churn", "msno"])
y = df["is_churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (776768, 13)
Test shape: (194192, 13)


## **5. Baseline Model Evaluation**

In [ ]:
# 5.1 DummyClassifier
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_pred = dummy.predict_proba(X_test)[:,1]
print("Baseline ROC-AUC:", roc_auc_score(y_test, dummy_pred))

Baseline ROC-AUC: 0.5


In [25]:
# 5.2 LogisticRegression

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer

# 1. Identify categorical and numeric columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

# 2. Preprocessing for numeric features: impute missing values
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# 3. Preprocessing for categorical features: impute + one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 4. Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# 5. Build the full pipeline
LogisticRegression(
    solver="saga",
    max_iter=3000,
    class_weight="balanced"
)

# 6. Fit the model
logreg_pipeline.fit(X_train, y_train)

# 7. Predict probabilities
logreg_pred = logreg_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression ROC-AUC:", roc_auc_score(y_test, logreg_pred))


Logistic Regression ROC-AUC: 0.9237980739210498


c:\Users\pauli\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [28]:
# 5.3 DecisionTreeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Identify categorical and numeric columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

# 2. Preprocessing for numeric features: impute missing values
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# 3. Preprocessing for categorical features: impute + one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 4. Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# 5. Build the full pipeline for Decision Tree
dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=3,
        class_weight="balanced",
        random_state=42
    ))
])

# 6. Fit the model
dt_pipeline.fit(X_train, y_train)

# 7. Predict probabilities
dt_pred = dt_pipeline.predict_proba(X_test)[:, 1]

print("Decision Tree ROC-AUC:", roc_auc_score(y_test, dt_pred))



Decision Tree ROC-AUC: 0.9425466092725916


In [29]:
# 5.4 Naive Bayes
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Identify categorical and numeric columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

# 2. Preprocessing for numeric features: impute missing values
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# 3. Preprocessing for categorical features: impute + one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 4. Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# 5. Build the full pipeline for Naive Bayes
nb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GaussianNB())
])

# 6. Fit the model
nb_pipeline.fit(X_train, y_train)

# 7. Predict probabilities
nb_pred = nb_pipeline.predict_proba(X_test)[:, 1]

print("Naive Bayes ROC-AUC:", roc_auc_score(y_test, nb_pred))

Naive Bayes ROC-AUC: 0.8457529109578916
